# Unsupervised ALBERT MLM Pre-training

This notebook demonstrates unsupervised pre-training of the ALBERT model on reaction SMILES data.

## Setup

1. **Install dependencies**: Run cell 0 to install `agave_chem`, `torch`, and `pandas`.
2. **Configure paths**: Update `TRAINING_DATA_FILE` and `SAVE_DIR` in cell 2 to point to your data and desired output directory.
3. **Tune hyperparameters**: Adjust `NUM_EPOCHS`, `BATCH_SIZE`, `WARMUP_STEPS`, etc. in cell 2.
4. **Resume from checkpoint**: Set `RESUME_FROM_CHECKPOINT` to a `.pt` checkpoint path to resume training.
5. **Early stopping**: Set `EARLY_STOPPING_PATIENCE` > 0 to stop training when validation loss stops improving.

## New features

- `save_best_model`: Saves the best model (by validation loss) to `{SAVE_DIR}/best_model.pt`
- `early_stopping_patience`: Stops training after N epochs without improvement (0 = disabled)
- `early_stopping_min_delta`: Minimum validation loss improvement to count as progress
- `resume_from_checkpoint`: Path to a `.pt` checkpoint to resume from
- `max_length`, `num_workers`, `prefetch_factor`, `masking_mode`: Now exposed as parameters

In [ ]:
!pip install git+https://github.com/denovochem/agave_chem.git -U --force-reinstall --no-cache-dir
!python -m pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install pandas

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from model_training_scripts.albert_mapper_unuspervised_training import (
    TrainingConfig,
    main,
)

In [ ]:
import os
import random

from rdkit import Chem, RDLogger

NUM_EPOCHS = 20
BATCH_SIZE = 64
WARMUP_STEPS = 10000
LOGGING_STEPS = 100
TRAIN_PCT = 0.99
MAX_LENGTH = 384
NUM_WORKERS = 8
PREFETCH_FACTOR = 4
MASKING_MODE = "span"
SEED = 42
SAVE_BEST_MODEL = True
EARLY_STOPPING_PATIENCE = 0
EARLY_STOPPING_MIN_DELTA = 0.0
RESUME_FROM_CHECKPOINT = None
TRAINING_DATA_FILE = "/workspace/data/uspto_all_reactions_training.txt"
SAVE_DIR = "/workspace/saved_models/albert-04-01-2026"
os.makedirs(SAVE_DIR, exist_ok=True)

RDLogger.DisableLog("rdApp.*")

In [ ]:
def canonicalize_smiles(
    smiles: str,
    isomeric: bool = True,
    remove_mapping: bool = True,
    canonicalize_tautomer: bool = True,
) -> str:
    """
    Converts SMILES strings to their canonical form using RDKit.

    Takes a SMILES string (potentially containing multiple fragments separated by periods),
    splits it into fragments, sorts them, and converts each to its canonical form. Handles
    atom mapping and isomeric SMILES options.

    Args:
        smiles (str): The input SMILES string to canonicalize
        isomeric (bool): Whether to retain isomeric information. Defaults to True
        remove_mapping (bool): Whether to remove atom mapping numbers. Defaults to True
        canonicalize_tautomer (bool): Whether to use the canonical tautomer. Defaults to True

    Returns:
        str: The canonicalized SMILES string. If conversion fails, returns the input string
            unchanged.
    """
    x = smiles.split(".")
    x = sorted(x)
    frags = []
    for i in x:
        m = Chem.MolFromSmiles(i)
        if canonicalize_tautomer:
            m = tautomer_enumerator.Canonicalize(m)
        if remove_mapping:
            [a.SetAtomMapNum(0) for a in m.GetAtoms()]
        canonical_smiles_string = str(
            Chem.MolToSmiles(m, canonical=True, isomericSmiles=isomeric)
        )
        frags.append(canonical_smiles_string)
    canonical_smiles_string = ".".join(i for i in sorted(frags))
    return canonical_smiles_string


def canonicalize_reaction_smiles(
    rxn_smiles: str,
    isomeric: bool = True,
    remove_mapping: bool = True,
    canonicalize_tautomer: bool = False,
    return_canonicalized_atom_mapping: bool = False,
) -> str:
    """
    Canonicalizes a reaction SMILES string using RDKit.

    Takes a reaction SMILES string (potentially containing multiple fragments separated by periods),
    splits it into fragments, sorts them, and converts each to its canonical form. Handles
    atom mapping and isomeric SMILES options.

    Args:
        rxn_smiles (str): The input reaction SMILES string to canonicalize
        isomeric (bool): Whether to retain isomeric information. Defaults to True
        remove_mapping (bool): Whether to remove atom mapping numbers. Defaults to True
        canonicalize_tautomer (bool): Whether to use the canonical tautomer. Defaults to False

    Returns:
        str: The canonicalized reaction SMILES string. If conversion fails, returns the input string
            unchanged.
    """
    split_roles = rxn_smiles.split(">>")
    reaction_list = []
    for x in split_roles:
        role_list = []
        if x == "":
            continue
        y = x.split(".")
        for z in y:
            canonical_smiles = canonicalize_smiles(
                z,
                isomeric=isomeric,
                remove_mapping=remove_mapping,
                canonicalize_tautomer=canonicalize_tautomer,
            )
            role_list.append(canonical_smiles)

        role_list = sorted(role_list)
        role_list = [ele for ele in role_list if ele != ""]
        reaction_list.append(role_list)

    canonical_rxn_components = [".".join(role_list) for role_list in reaction_list]
    canonical_rxn = ">>".join(canonical_rxn_components)
    if return_canonicalized_atom_mapping:
        canonical_rxn = canonicalize_atom_mapping(canonical_rxn)
    return canonical_rxn

In [ ]:
rxns = []
with open(TRAINING_DATA_FILE, "r") as handle:
    i = 0
    for line in handle:
        if i % 10000 == 0:
            print(i)
        i += 1
        try:
            rxns.append(canonicalize_reaction_smiles(line.strip().replace("~", ".")))
        except Exception:
            print(f"Cannot canonicalize {i}")

rxns = list(set(rxns))
random.seed(42)
random.shuffle(rxns)
rxns_train = rxns[: int(len(rxns) * 0.99)]
rxns_val = rxns[int(len(rxns) * 0.99) :]

In [ ]:
training_config = TrainingConfig(
    output_dir=SAVE_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    seed=SEED,
    save_best_model=SAVE_BEST_MODEL,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
)

main(
    train_texts=rxns_train,
    val_texts=rxns_val,
    training_config=training_config,
    max_length=MAX_LENGTH,
    num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    masking_mode=MASKING_MODE,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)